# Blog Writing Agent utilizing NKit & Gemini

This notebook demonstrates how to build a production-ready Web Research & Blog Writing Agent using the locally modified NKit framework and the Gemini API.

In [ ]:
import os
import sys
import logging
from IPython.display import display, Markdown

# 1. Find the robust path to the project root containing the source code
current_dir = os.getcwd()
while current_dir and not os.path.isdir(os.path.join(current_dir, 'NKit')):
    parent = os.path.dirname(current_dir)
    if parent == current_dir: break
    current_dir = parent

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Using NKit repository at: {current_dir}")

# 2. Import EXACTLY using the capitalized folder name 'NKit' to bypass PyPI's 'nkit'
from NKit import Agent, Tool, LiveObserver
from NKit.llms import GeminiLLM
from NKit.safety import SafetyGate
from NKit.audit import WhyLog

# Suppress verbose HTTP logs if desired
logging.getLogger("urllib3").setLevel(logging.WARNING)

## Configuration
Provide your Gemini API Key.

In [ ]:
# os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

if "GEMINI_API_KEY" not in os.environ:
    print("⚠️ Please set GEMINI_API_KEY to continue.")

## Define Blog Writing Tools
We equip the agent with tools to write blog posts to a file system. NKit automatically handles wrapping this function safely.

In [ ]:
def save_blog_post(title: str, content: str) -> str:
    """
    Saves the generated blog post to a text file in the output directory.
    
    Args:
        title: The title of the blog post (used as filename)
        content: The markdown content of the blog
    """
    filename = f"./blog_drafts/{title.replace(' ', '_').lower()}.md"
    os.makedirs("./blog_drafts", exist_ok=True)
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Successfully saved blog post to {filename}"

save_tool = Tool(
    name="save_blog_post", 
    func=save_blog_post, 
    desc="Saves blog post content to a markdown file given a title and content."
)

## Initialize NKit Components
Adhering to the NKit plugin architecture, we inject the `GeminiLLM`, `SafetyGate` (for sandboxing), and the `LiveObserver` directly into the agent.

In [ ]:
llm = GeminiLLM(model="gemini-2.5-flash")
observer = LiveObserver()

# SafetyGate blocks any file writes outside of the allowed directories
safety_gate = SafetyGate(allowed_dirs=["./blog_drafts"])

# WhyLog creates an auditable JSONL trail answering 'what', 'why', and 'was it safe'
os.makedirs("./logs", exist_ok=True)
audit_log = WhyLog(path="./logs/blog_agent_audit.jsonl")

agent = Agent(
    llm=llm,
    observer=observer,
    safety_gate=safety_gate,
    why_log=audit_log
)

# Explicitly map to ToolRegistry
agent.registry.register(save_tool)
    
# Set up the observer to stream the agent's thought process into the notebook dynamically
@observer.on("agent.reasoning")
def print_thoughts(event):
    pass # You can un-comment to see thoughts in realtime
    # print(f"\n[THOUGHT]: {event.get('thought', '')}")

@observer.on("tool.before")
def print_action(event):
    print(f"\n► EXECUTING: {event['tool_name']} | WHY: {event.get('why', 'No reason provided')}")

## Execute the Mission!

In [ ]:
topic = "The future of Agentic AI frameworks in the Enterprise"
prompt = f"""
You are an expert technical blog writer.
Please write a comprehensive, engaging 500-word markdown blog post about: {topic}.
Once you have written it, use the 'save_blog_post' tool to save it to disk.
"""

try:
    result = agent.run(prompt)
    print("\n--- FINAL OUTPUT ---")
    display(Markdown(result))
except Exception as e:
    print(f"Execution Error: {e}")